# Colab C++ vs CUDA Diffraction Benchmark

Этот ноутбук предназначен для **Google Colab** и сравнивает две отдельные native-реализации из GitHub:
- `CPU C++` без CUDA
- `CUDA C++`

Важно: сам ноутбук, как и любой Colab/Jupyter, управляется Python-кодом, но измеряемые solver'ы здесь именно **нативные C++/CUDA executable**, а не Python-реализация.


## Что делает ноутбук

1. Клонирует или обновляет репозиторий с GitHub.
2. Собирает `CPU C++` версию отдельно.
3. При наличии GPU-runtime в Colab собирает `CUDA` версию.
4. Запускает обе версии на одинаковых наборах параметров.
5. Сохраняет таблицу результатов и строит графики по времени и speedup.


## Требования к runtime

- Для сравнения `CPU C++` и `CUDA` нужен **Colab GPU runtime**.
- Если GPU недоступен, ноутбук всё равно сможет собрать и гонять `CPU C++` baseline.
- По умолчанию используется ветка `feature/two-plates`; при необходимости можно поменять её в конфигурации.


In [ ]:
from __future__ import annotations

import itertools
import math
import statistics
import subprocess
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


In [ ]:
REPO_URL = 'https://github.com/MaximillianVoss/plate-diffraction-solver.git'
REPO_BRANCH = 'feature/two-plates'
WORKDIR = Path('/content/plate-diffraction-solver')
RESULTS_CSV = WORKDIR / 'benchmark_results_cpp_cuda.csv'


In [ ]:
def run_command(command, cwd: Path | None = None, check: bool = True):
    completed = subprocess.run(
        command,
        cwd=str(cwd) if cwd is not None else None,
        capture_output=True,
        text=True,
        shell=False,
    )
    if check and completed.returncode != 0:
        raise RuntimeError(
            f'Command failed: {command}\nSTDOUT:\n{completed.stdout}\nSTDERR:\n{completed.stderr}'
        )
    return completed


def ensure_repo():
    if WORKDIR.exists():
        run_command(['git', '-C', str(WORKDIR), 'fetch', 'origin'])
        run_command(['git', '-C', str(WORKDIR), 'checkout', REPO_BRANCH])
        run_command(['git', '-C', str(WORKDIR), 'pull', '--ff-only', 'origin', REPO_BRANCH])
    else:
        run_command(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(WORKDIR)])
    return WORKDIR


def detect_cuda_arch(default='75'):
    if shutil.which('nvidia-smi') is None:
        return None

    result = run_command(
        ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
        check=False,
    )
    if result.returncode != 0:
        return None

    value = result.stdout.strip().splitlines()[0].strip()
    if not value:
        return default
    return value.replace('.', '')


In [ ]:
repo_root = ensure_repo()
print('repo_root =', repo_root)
print(run_command(['git', '-C', str(repo_root), 'rev-parse', '--short', 'HEAD']).stdout.strip())


In [ ]:
CPU_BUILD_SCRIPT = repo_root / 'Diffraction.Cpp' / 'build_cpu.sh'
CPU_EXE = repo_root / 'Diffraction.Cpp' / 'build' / 'DiffractionCpu'
CUDA_BUILD_SCRIPT = repo_root / 'Diffraction.Cuda' / 'build_cuda.sh'
CUDA_EXE = repo_root / 'Diffraction.Cuda' / 'build' / 'DiffractionCuda'

cpu_build = run_command(['bash', str(CPU_BUILD_SCRIPT)], cwd=repo_root)
print(cpu_build.stdout)

cuda_arch = detect_cuda_arch()
cuda_available = cuda_arch is not None
print('cuda_available =', cuda_available, 'cuda_arch =', cuda_arch)

if cuda_available:
    cuda_build = run_command(['bash', str(CUDA_BUILD_SCRIPT), cuda_arch], cwd=repo_root, check=False)
    print(cuda_build.stdout or cuda_build.stderr)
    cuda_available = cuda_build.returncode == 0 and CUDA_EXE.exists()
    if not cuda_available:
        print('CUDA build failed, benchmark will continue with CPU only.')
else:
    print('GPU runtime not detected, benchmark will continue with CPU only.')


In [ ]:
def parse_solver_output(text: str):
    parsed = {
        'status': None,
        'backend': None,
        'assembly_ms': None,
        'solve_ms': None,
        'total_ms': None,
        'coefficients': [],
        'message': None,
    }
    coeff_map = {}
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line or '=' not in line:
            continue
        key, value = line.split('=', 1)
        key = key.strip()
        value = value.strip()
        if key.startswith('coeff_'):
            idx = int(key.split('_', 1)[1])
            re_text, im_text = value.split(',')
            coeff_map[idx] = complex(float(re_text), float(im_text))
        elif key in {'assembly_ms', 'solve_ms', 'total_ms'}:
            parsed[key] = float(value)
        else:
            parsed[key] = value

    parsed['coefficients'] = [coeff_map[i] for i in sorted(coeff_map)]
    return parsed


def build_solver_args(case: dict):
    theta_rad = math.radians(case['theta_deg'])
    return [
        '--alpha1', str(case['alpha1']),
        '--beta1', str(case['beta1']),
        '--alpha2', str(case['alpha2']),
        '--beta2', str(case['beta2']),
        '--lambda', str(case['lambda']),
        '--theta', repr(theta_rad),
        '--n', str(case['n']),
        '--skin-depth', str(case['skin_depth']),
    ]


def run_solver(executable: Path, case: dict):
    result = run_command([str(executable), *build_solver_args(case)], check=False)
    parsed = parse_solver_output(result.stdout)
    parsed['returncode'] = result.returncode
    parsed['stdout'] = result.stdout
    parsed['stderr'] = result.stderr
    return parsed


def max_coeff_diff(cpu_coeffs, gpu_coeffs):
    if not cpu_coeffs or not gpu_coeffs or len(cpu_coeffs) != len(gpu_coeffs):
        return None
    return max(abs(a - b) for a, b in zip(cpu_coeffs, gpu_coeffs))


In [ ]:
DEFAULT_CASE = {
    'alpha1': -1.5,
    'beta1': -0.5,
    'alpha2': 0.5,
    'beta2': 1.5,
    'lambda': 1.0,
    'theta_deg': 10.0,
    'n': 10,
    'skin_depth': 0.001,
}

cpu_sample = run_solver(CPU_EXE, DEFAULT_CASE)
print('CPU sample:', cpu_sample['status'], cpu_sample['backend'], cpu_sample['total_ms'])

if cuda_available:
    cuda_sample = run_solver(CUDA_EXE, DEFAULT_CASE)
    print('CUDA sample:', cuda_sample['status'], cuda_sample['backend'], cuda_sample['total_ms'])
    print('max_coeff_abs_diff =', max_coeff_diff(cpu_sample['coefficients'], cuda_sample['coefficients']))
else:
    print('CUDA sample skipped.')


In [ ]:
N_VALUES = [5, 10, 20, 40, 60]
ANGLE_VALUES_DEG = [0.0, 30.0, 60.0, 90.0]
SKIN_DEPTH_VALUES = [0.0, 0.001, 0.01, 0.1]
REPEATS = 2


In [ ]:
def benchmark_case(case: dict, repeats: int = REPEATS):
    rows = []

    cpu_runs = [run_solver(CPU_EXE, case) for _ in range(repeats)]
    rows.append({
        **case,
        'backend': 'CPU C++',
        'assembly_ms_mean': statistics.mean(run['assembly_ms'] for run in cpu_runs),
        'solve_ms_mean': statistics.mean(run['solve_ms'] for run in cpu_runs),
        'total_ms_mean': statistics.mean(run['total_ms'] for run in cpu_runs),
        'status': cpu_runs[-1]['status'],
        'max_coeff_abs_diff_vs_cpu': 0.0,
    })

    if cuda_available:
        cuda_runs = [run_solver(CUDA_EXE, case) for _ in range(repeats)]
        rows.append({
            **case,
            'backend': 'CUDA',
            'assembly_ms_mean': statistics.mean(run['assembly_ms'] for run in cuda_runs),
            'solve_ms_mean': statistics.mean(run['solve_ms'] for run in cuda_runs),
            'total_ms_mean': statistics.mean(run['total_ms'] for run in cuda_runs),
            'status': cuda_runs[-1]['status'],
            'max_coeff_abs_diff_vs_cpu': max_coeff_diff(cpu_runs[-1]['coefficients'], cuda_runs[-1]['coefficients']),
        })

    return rows


cases = []
for n, theta_deg, skin_depth in itertools.product(N_VALUES, ANGLE_VALUES_DEG, SKIN_DEPTH_VALUES):
    case = dict(DEFAULT_CASE)
    case['n'] = n
    case['theta_deg'] = theta_deg
    case['skin_depth'] = skin_depth
    cases.append(case)

benchmark_rows = []
for index, case in enumerate(cases, start=1):
    print(f'[{index}/{len(cases)}] n={case["n"]}, theta={case["theta_deg"]}, skin={case["skin_depth"]}')
    benchmark_rows.extend(benchmark_case(case))

benchmark_df = pd.DataFrame(benchmark_rows)
benchmark_df.to_csv(RESULTS_CSV, index=False)
print('Saved:', RESULTS_CSV)
benchmark_df.head()


## Обозначения столбцов в результатах

Ниже используются такие поля:

- `backend`: какая реализация запускалась, `CPU C++` или `CUDA`.
- `alpha1`, `beta1`, `alpha2`, `beta2`: границы двух пластин.
- `lambda`: длина волны.
- `theta_deg`: угол падения в градусах.
- `n`: параметр усечения, размер локального базиса на одну пластину.
- `skin_depth`: толщина скин-слоя.
- `assembly_ms_mean`: среднее время сборки матрицы и правой части за несколько повторов.
- `solve_ms_mean`: среднее время решения СЛАУ за несколько повторов.
- `total_ms_mean`: среднее полное время backend'а.
- `status`: статус запуска backend'а.
- `max_coeff_abs_diff_vs_cpu`: максимальное абсолютное расхождение коэффициентов относительно `CPU C++` для того же набора параметров.

Если позже backend'ы начнут отдавать дополнительные диагностические метрики, их можно будет включить в тот же CSV по аналогии.


## Таблица 1. Сводка средних времен по backend и N

Таблица показывает усреднённые времена `assembly`, `solve` и `total` для каждого backend'а при разных `N`.
Смысл: быстро увидеть масштаб времени расчёта и как он меняется с ростом размерности задачи.


In [ ]:
summary = (
    benchmark_df
    .groupby(['backend', 'n'])[['assembly_ms_mean', 'solve_ms_mean', 'total_ms_mean']]
    .mean()
    .reset_index()
)
summary


## Таблица 2. Попарное сравнение CPU C++ и CUDA

Таблица строится только если `CUDA` доступна.
Она соединяет строки `CPU C++` и `CUDA` для одинаковых параметров и показывает:

- времена обеих реализаций,
- speedup как отношение `CPU_time / CUDA_time`,
- максимальное расхождение коэффициентов между `CPU C++` и `CUDA`.

Если `speedup > 1`, то CUDA быстрее CPU по соответствующей метрике.


In [ ]:
merged = None
if cuda_available:
    cpu_df = benchmark_df[benchmark_df['backend'] == 'CPU C++'].copy()
    cuda_df = benchmark_df[benchmark_df['backend'] == 'CUDA'].copy()
    merged = cpu_df.merge(
        cuda_df,
        on=['alpha1', 'beta1', 'alpha2', 'beta2', 'lambda', 'theta_deg', 'n', 'skin_depth'],
        suffixes=('_cpu', '_cuda')
    )
    merged['speedup_total_cpu_over_cuda'] = merged['total_ms_mean_cpu'] / merged['total_ms_mean_cuda']
    merged['speedup_solve_cpu_over_cuda'] = merged['solve_ms_mean_cpu'] / merged['solve_ms_mean_cuda']
    merged['speedup_assembly_cpu_over_cuda'] = merged['assembly_ms_mean_cpu'] / merged['assembly_ms_mean_cuda']
    merged[[
        'theta_deg', 'n', 'skin_depth',
        'assembly_ms_mean_cpu', 'assembly_ms_mean_cuda', 'speedup_assembly_cpu_over_cuda',
        'solve_ms_mean_cpu', 'solve_ms_mean_cuda', 'speedup_solve_cpu_over_cuda',
        'total_ms_mean_cpu', 'total_ms_mean_cuda', 'speedup_total_cpu_over_cuda',
        'max_coeff_abs_diff_vs_cpu_cuda'
    ]]
else:
    print('CUDA data not available, pairwise comparison table skipped.')


## Графики 1-3. Время сборки, решения и полного backend'а

Каждый график показывает зависимость времени от `N`.

- Ось `X`: `N`
- Ось `Y`: соответствующая временная метрика в миллисекундах
- Отдельные кривые: `CPU C++` и `CUDA`
- Отдельные subplot'ы: разные углы `theta_deg`
- Фиксированное значение `skin_depth` задаётся через `PLOT_SKIN_DEPTH`


In [ ]:
PLOT_SKIN_DEPTH = 0.001
PLOT_THETA_FOR_HEATMAP = 30.0
PLOT_METRICS = [
    ('assembly_ms_mean', 'Assembly time vs N'),
    ('solve_ms_mean', 'Solve time vs N'),
    ('total_ms_mean', 'Total backend time vs N'),
]


def plot_metric_by_theta(dataframe, metric_name: str, title: str, skin_depth: float):
    subset = dataframe[dataframe['skin_depth'] == skin_depth].copy()
    theta_values = sorted(subset['theta_deg'].unique())
    if not theta_values:
        print(f'No data for skin_depth={skin_depth}')
        return

    fig, axes = plt.subplots(1, len(theta_values), figsize=(5 * len(theta_values), 4), sharey=True)
    if len(theta_values) == 1:
        axes = [axes]

    for ax, theta_deg in zip(axes, theta_values):
        theta_slice = subset[subset['theta_deg'] == theta_deg]
        for backend in sorted(theta_slice['backend'].unique()):
            part = theta_slice[theta_slice['backend'] == backend].sort_values('n')
            ax.plot(part['n'], part[metric_name], marker='o', label=backend)
        ax.set_title(f'theta={theta_deg}°, skin={skin_depth}')
        ax.set_xlabel('N')
        ax.grid(True, alpha=0.3)

    axes[0].set_ylabel(metric_name)
    axes[-1].legend()
    fig.suptitle(title, y=1.05)
    plt.tight_layout()
    plt.show()


for metric_name, title in PLOT_METRICS:
    plot_metric_by_theta(benchmark_df, metric_name, title, PLOT_SKIN_DEPTH)


## График 4. Ускорение CUDA относительно CPU C++

Показывается speedup по формуле `CPU_time / CUDA_time`.

- Если значение больше `1`, CUDA быстрее CPU.
- Если значение меньше `1`, CPU быстрее CUDA.

Ниже строятся три версии speedup:
- по `assembly`,
- по `solve`,
- по `total`.


In [ ]:
if merged is not None:
    speedup_specs = [
        ('speedup_assembly_cpu_over_cuda', 'Assembly speedup CPU/CUDA'),
        ('speedup_solve_cpu_over_cuda', 'Solve speedup CPU/CUDA'),
        ('speedup_total_cpu_over_cuda', 'Total speedup CPU/CUDA'),
    ]

    fig, axes = plt.subplots(1, len(speedup_specs), figsize=(18, 4), sharex=False)
    for ax, (column_name, title) in zip(axes, speedup_specs):
        subset = merged[merged['skin_depth'] == PLOT_SKIN_DEPTH].copy()
        for theta_deg in sorted(subset['theta_deg'].unique()):
            part = subset[subset['theta_deg'] == theta_deg].sort_values('n')
            ax.plot(part['n'], part[column_name], marker='o', label=f'theta={theta_deg}°')
        ax.axhline(1.0, color='black', linestyle='--', linewidth=1)
        ax.set_title(title)
        ax.set_xlabel('N')
        ax.set_ylabel('CPU_time / CUDA_time')
        ax.grid(True, alpha=0.3)
    axes[-1].legend()
    plt.tight_layout()
    plt.show()
else:
    print('Speedup plots skipped: CUDA data not available.')


## Графики 5-6. Тепловые карты

Тепловые карты нужны, чтобы быстрее увидеть области параметров, где backend работает медленнее или быстрее.

Сейчас строятся две карты:
- `N × theta_deg` для фиксированного `skin_depth`
- `N × skin_depth` для фиксированного `theta_deg`

Величина на карте: `total_ms_mean` для выбранного backend'а.
Если доступна CUDA, дополнительно строится карта `speedup_total_cpu_over_cuda`.


In [ ]:
def plot_heatmap(dataframe, index_col, column_col, value_col, title, fmt='.2f', cmap='viridis'):
    pivot = dataframe.pivot(index=index_col, columns=column_col, values=value_col).sort_index().sort_index(axis=1)
    fig, ax = plt.subplots(figsize=(6, 4))
    image = ax.imshow(pivot.values, aspect='auto', cmap=cmap, origin='lower')
    ax.set_title(title)
    ax.set_xlabel(column_col)
    ax.set_ylabel(index_col)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([str(v) for v in pivot.columns])
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([str(v) for v in pivot.index])

    for y in range(len(pivot.index)):
        for x in range(len(pivot.columns)):
            value = pivot.values[y, x]
            if pd.notna(value):
                ax.text(x, y, format(value, fmt), ha='center', va='center', color='white', fontsize=8)

    fig.colorbar(image, ax=ax)
    plt.tight_layout()
    plt.show()


cpu_total = benchmark_df[benchmark_df['backend'] == 'CPU C++'].copy()
plot_heatmap(
    cpu_total[cpu_total['skin_depth'] == PLOT_SKIN_DEPTH],
    index_col='n',
    column_col='theta_deg',
    value_col='total_ms_mean',
    title=f'CPU C++ total_ms_mean, skin={PLOT_SKIN_DEPTH}'
)
plot_heatmap(
    cpu_total[cpu_total['theta_deg'] == PLOT_THETA_FOR_HEATMAP],
    index_col='n',
    column_col='skin_depth',
    value_col='total_ms_mean',
    title=f'CPU C++ total_ms_mean, theta={PLOT_THETA_FOR_HEATMAP}°'
)

if cuda_available:
    cuda_total = benchmark_df[benchmark_df['backend'] == 'CUDA'].copy()
    plot_heatmap(
        cuda_total[cuda_total['skin_depth'] == PLOT_SKIN_DEPTH],
        index_col='n',
        column_col='theta_deg',
        value_col='total_ms_mean',
        title=f'CUDA total_ms_mean, skin={PLOT_SKIN_DEPTH}'
    )
    plot_heatmap(
        cuda_total[cuda_total['theta_deg'] == PLOT_THETA_FOR_HEATMAP],
        index_col='n',
        column_col='skin_depth',
        value_col='total_ms_mean',
        title=f'CUDA total_ms_mean, theta={PLOT_THETA_FOR_HEATMAP}°'
    )

if merged is not None:
    plot_heatmap(
        merged[merged['skin_depth'] == PLOT_SKIN_DEPTH],
        index_col='n',
        column_col='theta_deg',
        value_col='speedup_total_cpu_over_cuda',
        title=f'Speedup total CPU/CUDA, skin={PLOT_SKIN_DEPTH}',
        cmap='magma'
    )
    plot_heatmap(
        merged[merged['theta_deg'] == PLOT_THETA_FOR_HEATMAP],
        index_col='n',
        column_col='skin_depth',
        value_col='speedup_total_cpu_over_cuda',
        title=f'Speedup total CPU/CUDA, theta={PLOT_THETA_FOR_HEATMAP}°',
        cmap='magma'
    )


## График 7. Расхождение коэффициентов между CPU C++ и CUDA

Этот график показывает не производительность, а численное совпадение решений:

- Ось `X`: `N`
- Ось `Y`: `max |cpu - cuda|` по коэффициентам
- Разные кривые: разные `theta_deg`
- Шкала `Y`: логарифмическая

Чем ниже кривая, тем ближе `CUDA` к `CPU C++` на одинаковых параметрах.


In [ ]:
if merged is not None:
    diff_subset = merged[merged['skin_depth'] == PLOT_SKIN_DEPTH].copy()
    fig, ax = plt.subplots(figsize=(8, 4))
    for theta_deg in sorted(diff_subset['theta_deg'].unique()):
        part = diff_subset[diff_subset['theta_deg'] == theta_deg].sort_values('n')
        ax.plot(part['n'], part['max_coeff_abs_diff_vs_cpu_cuda'], marker='o', label=f'theta={theta_deg}°')
    ax.set_title(f'Max coefficient abs diff, skin={PLOT_SKIN_DEPTH}')
    ax.set_xlabel('N')
    ax.set_ylabel('max |cpu - cuda|')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()
else:
    print('Coefficient-diff plot skipped: CUDA data not available.')


## Дальше

- Если нужен более плотный sweep, увеличить `REPEATS`, `N_VALUES`, `ANGLE_VALUES_DEG` и `SKIN_DEPTH_VALUES`.
- Если потребуется сравнение не только времени, но и точности/энергетики, можно расширить CLI-формат backend'ов дополнительными метриками.
- Для отчётных материалов удобно использовать `benchmark_results_cpp_cuda.csv` как источник таблиц и финальных графиков.
